# SMA Variable Voltage Controller — Log Visualization

Plots data captured over the serial console from the firmware
(`src/main.cpp`). Two kinds of capture are supported:

- **Cycle logs** — output of a `cycle` command. Columns:
  `t_ms, phase, target_v, wiper, vldo_v, isns_v, current_a`
- **Calibration tables** — output of `cal` or `caldump`. Columns: `code, vldo_v`

### How to capture
Open the serial monitor at 115200 baud and save the output to a `.csv` file,
e.g. with PlatformIO:

```
pio device monitor -b 115200 | tee logs/run1.csv
```

Lines beginning with `#` are informational and are ignored by the loader below.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Point these at your captured files:
CYCLE_CSV = "logs/run1.csv"     # output of a `cycle` command
CAL_CSV   = "logs/cal.csv"      # output of `cal` / `caldump`

# Board constants (mirror include/config.h)
RSHUNT   = 0.33   # ohms
INA_GAIN = 10.0   # V/V (INA296A1)


## Loader
Reads a serial capture, skips `#` comment lines, and locates the CSV header row.

In [ ]:
def load_capture(path):
    """Load a serial capture, tolerating leading '#' comments and banner text.
    Returns a DataFrame parsed from the first CSV header line found."""
    rows = []
    header = None
    with open(path, "r", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            if header is None:
                # first non-comment, non-empty line is the header
                if "," in s and any(c.isalpha() for c in s):
                    header = [h.strip() for h in s.split(",")]
                continue
            rows.append(s.split(","))
    if header is None:
        raise ValueError(f"No CSV header found in {path}")
    df = pd.DataFrame(rows, columns=header)
    # numeric coercion for everything except obvious string columns
    for c in df.columns:
        if c == "phase":
            continue
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.dropna(how="all")


## Cycle log: V_LDO and SMA current vs. time
Phases are shaded (HEAT vs COOL).

In [ ]:
df = load_capture(CYCLE_CSV)
df["t_s"] = df["t_ms"] / 1000.0

# Recompute current from raw ISNS as a cross-check (matches firmware formula).
if "isns_v" in df:
    df["current_calc_a"] = df["isns_v"] / (INA_GAIN * RSHUNT)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(df["t_s"], df["vldo_v"], color="tab:blue", lw=1.2, label="V_LDO (measured)")
if "target_v" in df:
    ax1.plot(df["t_s"], df["target_v"], color="tab:gray", ls="--", lw=1.0, label="target V")
ax1.set_ylabel("V_LDO [V]")
ax1.legend(loc="upper right"); ax1.grid(alpha=0.3)

ax2.plot(df["t_s"], df["current_a"], color="tab:red", lw=1.2, label="I_SMA")
ax2.set_ylabel("SMA current [A]"); ax2.set_xlabel("time [s]")
ax2.legend(loc="upper right"); ax2.grid(alpha=0.3)

# Shade HEAT phases
if "phase" in df:
    in_heat = (df["phase"] == "HEAT").values
    t = df["t_s"].values
    start = None
    for i, h in enumerate(in_heat):
        if h and start is None:
            start = t[i]
        elif not h and start is not None:
            for ax in (ax1, ax2):
                ax.axvspan(start, t[i], color="orange", alpha=0.12)
            start = None
    if start is not None:
        for ax in (ax1, ax2):
            ax.axvspan(start, t[-1], color="orange", alpha=0.12)

fig.suptitle("SMA cycle — orange = HEAT phase")
fig.tight_layout()
plt.show()


## Quick stats per phase

In [ ]:
if "phase" in df:
    summary = df.groupby("phase").agg(
        n=("t_ms", "size"),
        vldo_mean=("vldo_v", "mean"),
        vldo_max=("vldo_v", "max"),
        current_mean=("current_a", "mean"),
        current_max=("current_a", "max"),
    )
    display(summary)


## Calibration curve: wiper code vs. V_LDO
Run a `cal` (or `caldump`) capture into `CAL_CSV`.

In [ ]:
try:
    cal = load_capture(CAL_CSV)
    plt.figure(figsize=(9, 5))
    plt.plot(cal["code"], cal["vldo_v"], marker=".", ms=4, lw=1, color="tab:green")
    plt.xlabel("wiper code (0-127)"); plt.ylabel("V_LDO [V]")
    plt.title("MCP4018 wiper code -> V_LDO (TPS7A57)")
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(f"V_LDO range: {cal['vldo_v'].min():.3f} .. {cal['vldo_v'].max():.3f} V")
except FileNotFoundError:
    print(f"No calibration capture at {CAL_CSV} yet. Run `cal` and save the output there.")
